In [1]:
import os
import glob

import json
import rasterio
from rasterio.mask import mask
from rasterio.windows import Window
import numpy as np 
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon,box

In [2]:
FILEPATH = r'C:\Users\b1114674\Desktop\thesis\AI4EO_data\international-charter-eo4ai-etq-challenge\Training'

EXCELPATH = r'C:\Users\b1114674\Desktop\thesis\AI4EO_data\international-charter-eo4ai-etq-challenge\Training\Phase_1_Dataset_Details.xlsx'

DESTINATION_PRE_PATH = r'C:\Users\b1114674\Desktop\thesis\data\TRAINING_DATA\PRE'
DESTINATION_POST_PATH = r'C:\Users\b1114674\Desktop\thesis\data\TRAINING_DATA\POST'

In [ ]:
class GEO_PreProssessing:

    def __init__(self, FILEPATH,EXCELPATH,DESTINATION_PRE_PATH,DESTINATION_POST_PATH,chip_size = (224,224,3)):
        
        self.filepath = FILEPATH
        self.excelpath = EXCELPATH

        self.pre_destination = DESTINATION_PRE_PATH
        self.post_destination = DESTINATION_POST_PATH

        self.chip_size = chip_size

        collector = PathCollector(self.filepath,self.excelpath)
        self.gdf = collector.collect()

    def open(self,path,no_channels=[1,2,3],transpose=True):
        '''
        Read the image 

        Args:
        path(str): filepath to the image
        no_channels(list(int)): how many channels do we need
        transpose(bool): True to change the shape of image to (H,W,C)

        Return: Image in numpy array format as (H,W,C) and Source format
        '''
        src = rasterio.open(path)
        img = src.read(no_channels)
        if transpose:
              img.transpose(1,2,0)
              return img,src
        else:
              return img,src

    def findOverlap(self,in_pre,in_post,out_pre,out_post,no_channels):
        
        '''
        Find the overlapping area an save both raster within intersection extent

        Args:
        in_Pre(str) : Absolute filepath to the pre image file
        in_post(str) : Absolute filepath to the post image file
        out_pre(str) : Absolute filepath to the Save cropped pre image file
        out_post(str) : Absolute filepath to the Save cropped post image file
        no_channels(list(int)): how many channels do we need
        
        '''

        _,pre_src = self.open(in_pre,no_channels)
        pre_bounds = box(*pre_src.bounds)
        crs = pre_src.crs
        
        _,post_src = self.open(in_post,no_channels)
        post_bounds = box(*post_src.bounds)

        common_area = pre_bounds.intersection(post_bounds)
    

        assert not common_area.is_empty, f'No Overlapping areas in the given images {os.path.basename(in_pre)} and {os.path.basename(in_post)}'

        # Converts Shapely geometry to GeoJSON format
        common_geojson = [json.loads(gpd.GeoSeries([common_area], crs=crs).to_json())['features'][0]['geometry']]

        pre_clip, pre_transform = mask(pre_src, common_geojson, crop=True)
        pre_meta = pre_src.meta.copy()
        pre_meta.update({
            "height": pre_clip.shape[1],
            "width": pre_clip.shape[2],
            "count": pre_clip.shape[0],
            "transform": pre_transform
            })
    

        post_clip, post_transform = mask(post_src, common_geojson, crop=True)
        post_meta = post_src.meta.copy()
        post_meta.update({
            "height": post_clip.shape[1],
            "width": post_clip.shape[2],
            "count": post_clip.shape[0],
            "transform": post_transform
            })
        
        try:
            with rasterio.open(out_pre,'w',**pre_meta) as pre_dest:
                                pre_dest.write(pre_clip)

            with rasterio.open(out_post,'w',**post_meta) as post_dest:
                                post_dest.write(post_clip)

        except ValueError:
              print('Invalid filepath!')


    def makeChips(self,in_pre,in_post,out_pre,out_post,no_channels,chip_size=224,stride=224):    

        '''

        Args:
        in_Pre(str) : Absolute filepath to the pre image file
        in_post(str) : Absolute filepath to the post image file
        out_pre(str) : Absolute filepath to the Save cropped pre image file
        out_post(str) : Absolute filepath to the Save cropped post image file
        no_channels(list(int)): how many channels do we need
        chip_size(int): The size of the image height and width
        stride(int): step size or overlap between the chips
        
        '''
        with rasterio.open(in_pre) as pre_src, rasterio.open(in_post) as post_src:
              for i in range(0,pre_src.height,stride):
                    for j in range(0,pre.height,stride):
                        window = Window(i,j,chip_size,chip_size)

                        pre_img = pre_src.read(no_channels,window=window)
                        post_img = post_src.read(no_channels,window=window)

                        pre_meta = pre_src.meta.copy()
                        pre_meta.update({
                            "height":pre_img.shape[1],
                            "widht":pre_img.shape[2],
                            "count":pre_img.shape[0],
                            "transform":rasterio.windows.transform(window,pre_src.transform)  
                        })

                        post_meta = post_src.meta.copy()
                        post_meta.update({
                            "height":post_img.shape[1],
                            "widht":post_img.shape[2],
                            "count":post_img.shape[0],
                            "transform":rasterio.windows.transform(window,post_src.transform)  
                        })
                        try:

                            if os.path.exists(out_pre) and os.path.exists(out_post):

                                with rasterio.open(out_pre,"w",**pre_meta) as pre_dest:
                                    pre_dest.write(pre_img)

                                with rasterio.open(out_post,"w",**post_meta) as post_dest:
                                    post_dest.write(post_img)

                        except ValueError:
                             print("Invalid path!")

    def Mean_Std(self,IMAGE_FOLDER_PATH,no_channels):
        '''
        Computing the Mean and Std for the whole dataset

        Args:
            IMAGE_FOLDER_PATH(str): Provide the directory consist of images (SAMPLE: ./ROOT_DATA_FOLDER/PRE_FOLDER)
        '''
        
        try:

            for path in os.listdir(IMAGE_FOLDER_PATH):
                img,_= open(path,no_channels,transpose=False)
                pixels = img.reshape(img.shape[0], -1).T 

                mean = np.zeros(pixels.shape[1])
                M2 = np.zeros(pixels.shape[1])

                n = pixels.shape[0]
                new_mean = np.mean(pixels, axis=0)
                new_M2 = np.var(pixels, axis=0) * n

                delta = new_mean - mean
                total_n = n_pixels + n
                mean += delta * n / total_n
                M2 += new_M2 + delta**2 * n_pixels * n / total_n
                n_pixels = total_n

            std = np.sqrt(M2 / n_pixels)
            print(f'[MEAN]----{mean}')
            print(f'[STD]----{std}')
        
        except ValueError:
            print('PATH is invalid !')

In [16]:
process = GEO_PreProssessing(FILEPATH,EXCELPATH,DESTINATION_PRE_PATH,DESTINATION_POST_PATH)

In [17]:
img,src= process.open(r'C:\Users\b1114674\Desktop\thesis\data\TRAINING_DATA\PRE\pre3.tif')
img

array([[[65535, 65535, 65535],
        [65535, 65535, 65535],
        [65535, 65535, 65535],
        ...,
        [65535, 65535, 65535],
        [65535, 65535, 65535],
        [65535, 65535, 65535]],

       [[65535, 65535, 65535],
        [65535, 65535, 65535],
        [65535, 65535, 65535],
        ...,
        [65535, 65535, 65535],
        [65535, 65535, 65535],
        [65535, 65535, 65535]],

       [[65535, 65535, 65535],
        [65535, 65535, 65535],
        [65535, 65535, 65535],
        ...,
        [65535, 65535, 65535],
        [65535, 65535, 65535],
        [65535, 65535, 65535]],

       ...,

       [[65535, 65535, 65535],
        [65535, 65535, 65535],
        [65535, 65535, 65535],
        ...,
        [65535, 65535, 65535],
        [65535, 65535, 65535],
        [65535, 65535, 65535]],

       [[65535, 65535, 65535],
        [65535, 65535, 65535],
        [65535, 65535, 65535],
        ...,
        [65535, 65535, 65535],
        [65535, 65535, 65535],
        [65535,